In [ ]:
# Pix2Pix Attention visualization every 10 epochs
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
import torchvision.transforms as transforms

from models import networks
from models.dino_attention import DinoAttentionExtractor


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE = 224
DINO_SIZE = 224
NGF = 48
EPOCH_STRIDE = 10
MAX_EPOCH = 100
EXP_NAME = "pix2pix_attention_attn_v1_large"
CHECKPOINT_DIR = Path(f"/home/ljc/code/PaBoT-main/checkpoints/{EXP_NAME}")
IMG_PATH_CT = "/home/ljc/code/PaBoT-main/datasets/testA/1BA278_slice192.png"
IMG_PATH_MRI = "/home/ljc/code/PaBoT-main/datasets/testB/1BA278_slice192.png"


def preprocess_image(image_path, image_size=256):
    # Match pix2pix normalization [-1, 1]
    transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    ])
    img = Image.open(image_path).convert("RGB")
    x = transform(img).unsqueeze(0)
    return x


def normalize_to_01(feat, eps=1e-6):
    min_v = feat.amin(dim=(2, 3), keepdim=True)
    max_v = feat.amax(dim=(2, 3), keepdim=True)
    return (feat - min_v) / (max_v - min_v + eps)


def load_generator_checkpoint(ckpt_path, device):
    netG = networks.define_G(
        768,
        1,
        NGF,
        "resnet_6blocks",
        norm="instance",
        use_dropout=True,
        init_type="normal",
        init_gain=0.02,
        no_antialias=False,
        no_antialias_up=False,
        gpu_ids=[],
        opt=None,
    ).to(device).eval()
    state = torch.load(ckpt_path, map_location=device)
    missing = netG.load_state_dict(state, strict=True)
    if len(missing.missing_keys) > 0 or len(missing.unexpected_keys) > 0:
        print(f"Warning: missing={missing.missing_keys}, unexpected={missing.unexpected_keys}")
    return netG


print("Loading DINO and fixed images...")
dino_attn = DinoAttentionExtractor(model_name="dino_vitb8", image_size=DINO_SIZE).to(DEVICE).eval()
img_ct = preprocess_image(IMG_PATH_CT, IMG_SIZE).to(DEVICE)
img_mri = preprocess_image(IMG_PATH_MRI, IMG_SIZE).to(DEVICE)

epoch_list = list(range(EPOCH_STRIDE, MAX_EPOCH + 1, EPOCH_STRIDE))
with torch.no_grad():
    att_ct_real = dino_attn(img_ct)
    att_ct_real = normalize_to_01(att_ct_real)[0, 0].cpu().numpy()

rows = len(epoch_list)
fig, axes = plt.subplots(rows, 3, figsize=(14, 3.2 * rows))
if rows == 1:
    axes = np.expand_dims(axes, axis=0)

for row, epoch in enumerate(epoch_list):
    ckpt_path = CHECKPOINT_DIR / f"{epoch}_net_G.pth"
    if not ckpt_path.exists():
        print(f"[skip] missing checkpoint: {ckpt_path}")
        for c in range(3):
            axes[row, c].axis("off")
        continue

    print(f"[load] epoch {epoch}: {ckpt_path.name}")
    netG = load_generator_checkpoint(str(ckpt_path), device=DEVICE)

    with torch.no_grad():
        att_mri, feat_mri = dino_attn(img_mri, return_patch_feat=True)
        # Normalize features exactly as in Pix2PixAttentionModel._normalize_feature_map
        mean = feat_mri.mean(dim=(2, 3), keepdim=True)
        std = feat_mri.std(dim=(2, 3), keepdim=True, unbiased=False).clamp_min(1e-6)
        feat_mri_norm = (feat_mri - mean) / std
        
        fake_att_ct = netG(feat_mri_norm)
        att_mri_vis = normalize_to_01(att_mri)[0, 0].cpu().numpy()
        fake_att_ct_vis = normalize_to_01(fake_att_ct)[0, 0].cpu().numpy()

    axes[row, 0].imshow(att_mri_vis, cmap="jet", vmin=0.0, vmax=1.0)
    axes[row, 0].set_title(f"att_mri (epoch {epoch})", fontsize=10)
    axes[row, 0].axis("off")

    axes[row, 1].imshow(fake_att_ct_vis, cmap="jet", vmin=0.0, vmax=1.0)
    axes[row, 1].set_title(f"fake_att_ct (epoch {epoch})", fontsize=10)
    axes[row, 1].axis("off")

    axes[row, 2].imshow(att_ct_real, cmap="jet", vmin=0.0, vmax=1.0)
    axes[row, 2].set_title("att_ct_real", fontsize=10)
    axes[row, 2].axis("off")

plt.tight_layout()
plt.show()
print(f"generated visualizations for {len(epoch_list)} epochs from {EPOCH_STRIDE} to {MAX_EPOCH}")
